# Multi-Class News Classification with BERT (PyTorch + Hugging Face)

**Dataset:** AG News (World, Sports, Business, Sci/Tech)
**Model:** `bert-base-uncased` fine-tuned end to end

**Before you run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

Improvements over a basic tutorial version:
- Proper **train / validation / test** split (the test set is only used once, at the end)
- **Max length chosen from data** (not blindly 512) with dynamic padding
- **TF-IDF + Logistic Regression baseline** for comparison
- Linear warmup/decay schedule, mixed precision, gradient clipping, best-checkpoint saving
- Confusion matrix, classification report, error analysis and an inference function

In [ ]:
!pip -q install transformers datasets scikit-learn accelerate

In [ ]:
import random, time, numpy as np, torch
import matplotlib.pyplot as plt, seaborn as sns
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "NO GPU - enable one in Runtime settings")

# ---------------- Config ----------------
MODEL_NAME = "bert-base-uncased"
EPOCHS     = 3
BATCH_SIZE = 32
LR         = 2e-5
WARMUP     = 0.1
WEIGHT_DECAY = 0.01
USE_SUBSET = False      # True = quick run (~10k train samples) to test the pipeline
SUBSET_N   = 10000

## 1. Load data and create a proper train / validation / test split

In [ ]:
raw = load_dataset("ag_news")
class_names = raw["train"].features["label"].names   # ['World','Sports','Business','Sci/Tech']
id2label = {i: n for i, n in enumerate(class_names)}
label2id = {n: i for i, n in id2label.items()}

# 90/10 stratified split of the ORIGINAL training set -> train / validation. Original test set stays untouched.
split = raw["train"].train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
train_ds, val_ds, test_ds = split["train"], split["test"], raw["test"]

if USE_SUBSET:
    train_ds = train_ds.shuffle(seed=SEED).select(range(SUBSET_N))
    val_ds   = val_ds.shuffle(seed=SEED).select(range(SUBSET_N // 5))

print({"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)})
print(class_names)
print(train_ds[0])

In [ ]:
# Class balance check
import collections
for name, d in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    c = collections.Counter(d["label"])
    print(name, {id2label[k]: v for k, v in sorted(c.items())})

## 2. Baseline: TF-IDF + Logistic Regression
A simple model to compare against. If BERT can't beat this by a clear margin, it isn't worth its cost.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

t0 = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
Xtr = tfidf.fit_transform(train_ds["text"])
clf = LogisticRegression(max_iter=1000, C=5.0).fit(Xtr, train_ds["label"])

base_val  = accuracy_score(val_ds["label"],  clf.predict(tfidf.transform(val_ds["text"])))
base_test = accuracy_score(test_ds["label"], clf.predict(tfidf.transform(test_ds["text"])))
print(f"TF-IDF + LR  |  val acc: {base_val:.4f}  |  test acc: {base_test:.4f}  |  {time.time()-t0:.0f}s")

## 3. Tokenization: choose max length from the data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = train_ds.shuffle(seed=SEED).select(range(min(5000, len(train_ds))))
lengths = [len(x) for x in tokenizer(sample["text"], truncation=False)["input_ids"]]
for p in (50, 90, 95, 99, 100):
    print(f"p{p}: {int(np.percentile(lengths, p))} tokens")

plt.hist(lengths, bins=50); plt.xlabel("tokens per article"); plt.ylabel("count"); plt.title("Token length distribution"); plt.show()

In [ ]:
# Pick a length that covers ~99% of articles (capped at 512, BERT's limit)
MAX_LEN = int(min(512, np.ceil(np.percentile(lengths, 99) / 8) * 8))
print("MAX_LEN =", MAX_LEN)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

def prep(ds):
    ds = ds.map(tokenize, batched=True, remove_columns=["text"])
    return ds.rename_column("label", "labels")

train_tok, val_tok, test_tok = prep(train_ds), prep(val_ds), prep(test_ds)

# Dynamic padding: each batch is padded only to its longest sequence (faster than padding everything to MAX_LEN)
collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")
train_dl = DataLoader(train_tok, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collator)
val_dl   = DataLoader(val_tok,   batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
test_dl  = DataLoader(test_tok,  batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)

b = next(iter(train_dl))
print({k: tuple(v.shape) for k, v in b.items()})

## 4. Model
BERT encoder + a dropout and linear classification layer on the pooled `[CLS]` output (4 outputs). All weights are fine-tuned.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(class_names), id2label=id2label, label2id=label2id
).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(model.classifier)   # the new, randomly initialised head

## 5. Training loop
AdamW + linear warmup/decay, mixed precision (fp16), gradient clipping. The best epoch (by validation accuracy) is saved.

In [ ]:
no_decay = ["bias", "LayerNorm.weight"]
param_groups = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],     "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(param_groups, lr=LR)

total_steps = EPOCHS * len(train_dl)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * total_steps), total_steps)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def evaluate(dl):
    model.eval()
    preds, labels, loss_sum, n = [], [], 0.0, 0
    for batch in dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(**batch)
        loss_sum += out.loss.item() * len(batch["labels"]); n += len(batch["labels"])
        preds.append(out.logits.argmax(-1).cpu()); labels.append(batch["labels"].cpu())
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return loss_sum / n, accuracy_score(labels, preds), preds, labels

history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_acc, best_path = 0.0, "best_model.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, t0 = 0.0, time.time()
    for step, batch in enumerate(train_dl, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
        if step % 200 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_dl)}  loss {running/step:.4f}")

    val_loss, val_acc, _, _ = evaluate(val_dl)
    history["train_loss"].append(running / len(train_dl)); history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
    print(f"Epoch {epoch}/{EPOCHS} | train loss {running/len(train_dl):.4f} | val loss {val_loss:.4f} | val acc {val_acc:.4f} | {time.time()-t0:.0f}s")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), best_path)
        print(f"  saved best model (val acc {best_acc:.4f})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history["train_loss"], marker="o", label="train"); ax[0].plot(history["val_loss"], marker="o", label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["val_acc"], marker="o"); ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 6. Final evaluation on the untouched test set (run once)

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_loss, test_acc, test_preds, test_labels = evaluate(test_dl)

print(f"BERT test accuracy:        {test_acc:.4f}")
print(f"TF-IDF + LR test accuracy: {base_test:.4f}   (BERT gain: {100*(test_acc-base_test):+.2f} points)\n")
print(classification_report(test_labels, test_preds, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (test set)"); plt.show()

## 7. Error analysis: what does the model get wrong?

In [ ]:
wrong = np.where(test_preds != test_labels)[0]
print(f"{len(wrong)} misclassified out of {len(test_labels)}\n")
for i in random.sample(list(wrong), min(8, len(wrong))):
    print(f"TRUE: {id2label[test_labels[i]]:9s} | PRED: {id2label[test_preds[i]]:9s}\n  {test_ds[int(i)]['text'][:220]}\n")

## 8. Inference on new headlines

In [ ]:
@torch.no_grad()
def predict(texts, top_k=2):
    model.eval()
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
    probs = torch.softmax(model(**enc).logits.float(), dim=-1).cpu()
    results = []
    for p in probs:
        top = p.topk(top_k)
        results.append([(id2label[int(i)], round(float(s), 3)) for s, i in zip(top.values, top.indices)])
    return results

examples = [
    "Manchester City beat Arsenal 2-1 in a thrilling Premier League clash",
    "Apple unveils new chip promising faster AI performance on laptops",
    "Stocks tumble as central bank signals more interest rate hikes",
    "UN leaders meet to discuss ceasefire talks in the region",
]
for t, r in zip(examples, predict(examples)):
    print(t, "\n  ->", r, "\n")

## 9. Save the model (and optionally download it)

In [ ]:
model.save_pretrained("bert_agnews_model")
tokenizer.save_pretrained("bert_agnews_model")

# Reload check
from transformers import pipeline
clf_pipe = pipeline("text-classification", model="bert_agnews_model", device=0 if device.type == "cuda" else -1)
print(clf_pipe("NASA announces new mission to study Jupiter's moons"))

# Uncomment to download a zip, or copy to Google Drive for permanent storage
# !zip -rq bert_agnews_model.zip bert_agnews_model
# from google.colab import files; files.download("bert_agnews_model.zip")
# from google.colab import drive; drive.mount('/content/drive'); !cp -r bert_agnews_model /content/drive/MyDrive/